# Hướng dẫn chạy trên Google Colab
Chạy các cell bên dưới theo thứ tự để huấn luyện mô hình.

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install rasterio albumentations opencv-python torchsummary

# Mount Google Drive (nếu lưu data trên Drive)
from google.colab import drive
drive.mount('/content/drive')

## 1. Config (Cấu hình)

In [ ]:
"""
Cấu hình chung cho dự án Forest Segmentation - Attention UNet.
Chỉnh sửa các đường dẫn và tham số tại đây để chạy trên local.
"""
import os

# ===== Cấu hình dữ liệu =====
IS_TRAINING_RGB_DATA = False  # True: 3 kênh (RGB), False: 4 kênh (RGB+NDVI)
NUM_IN_CHANNELS = 3 if IS_TRAINING_RGB_DATA else 4

# ===== Đường dẫn dữ liệu (CHỈNH SỬA CHO LOCAL) =====
# Trên Colab: "/content/drive/MyDrive/RVB_NAMDINH_2025"
# Trên Local: thay bằng đường dẫn thực tế
BASE_DIR = "/content/drive/MyDrive/RVB_NAMDINH_2025"  # ĐƯỜNG DẪN TRÊN COLAB CỦA BẠN

IMAGES_DIR = os.path.join(BASE_DIR, "images")
MASKS_DIR = os.path.join(BASE_DIR, "masks")

TRAIN_IMAGES_DIR = os.path.join(BASE_DIR, "train", "images")
TRAIN_MASKS_DIR = os.path.join(BASE_DIR, "train", "masks")
VALID_IMAGES_DIR = os.path.join(BASE_DIR, "valid", "images")
VALID_MASKS_DIR = os.path.join(BASE_DIR, "valid", "masks")

CHECKPOINT_DIR = os.path.join(
    BASE_DIR, "checkpoints_RGB" if IS_TRAINING_RGB_DATA else "checkpoints_RGB_NDVI"
)

# ===== Tham số training =====
PATCH_SIZE = 256
STRIDE = 256
BATCH_SIZE = 8
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
PATIENCE = 50  # Early stopping patience
NUM_WORKERS = 2  # DataLoader workers (giảm xuống 0 nếu Windows báo lỗi)
SEED = 42

# ===== Tham số model =====
NUM_CLASSES = 1  # Binary segmentation: Rừng / Phi rừng


## 2. Dataset & DataLoader

In [ ]:
"""
Dataset & DataLoader cho Forest Segmentation.
Bao gồm: đọc ảnh GeoTIFF, sliding window, augmentation, preprocessing.
"""
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import rasterio
import cv2
import albumentations as albu

# from config import (
    IS_TRAINING_RGB_DATA, NUM_IN_CHANNELS,
    TRAIN_IMAGES_DIR, TRAIN_MASKS_DIR,
    VALID_IMAGES_DIR, VALID_MASKS_DIR,
    PATCH_SIZE, STRIDE, BATCH_SIZE, NUM_WORKERS, SEED,
)


# ===== Đọc ảnh & mask =====

def load_image_and_mask(image_path, mask_path, is_rgb_only=IS_TRAINING_RGB_DATA):
    """
    Đọc ảnh GeoTIFF (R,G,B,NDVI) và mask.
    Returns: (image [H,W,C], mask [H,W]) chuẩn hóa min-max về [0,1].
    """
    with rasterio.open(image_path) as src:
        img = src.read()  # [bands, H, W]
        if is_rgb_only:
            rgb = img[:3, :, :].astype('float32')
            rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
            image_data = np.transpose(rgb, (1, 2, 0))
        else:
            img = img.astype('float32')
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            image_data = np.transpose(img, (1, 2, 0))

    with rasterio.open(mask_path) as src:
        mask = src.read(1)

    return image_data, mask


# ===== Sliding window =====

def sliding_window(image, mask, patch_size=256, stride=256):
    """Chia ảnh và mask thành các patches bằng sliding window."""
    H, W = image.shape[:2]
    patches, mask_patches = [], []

    y_positions = list(range(0, H - patch_size + 1, stride))
    x_positions = list(range(0, W - patch_size + 1, stride))

    if y_positions[-1] + patch_size < H:
        y_positions.append(H - patch_size)
    if x_positions[-1] + patch_size < W:
        x_positions.append(W - patch_size)

    for y in y_positions:
        for x in x_positions:
            patches.append(image[y:y+patch_size, x:x+patch_size])
            mask_patches.append(mask[y:y+patch_size, x:x+patch_size])

    return patches, mask_patches


# ===== Preprocessing (to_tensor, normalize) =====

def to_tensor(x, **kwargs):
    if x.ndim == 2:
        x = np.expand_dims(x, axis=-1)
    return x.transpose(2, 0, 1).astype('float32')


def preprocess_input(x, mean=None, std=None, input_range=None, **kwargs):
    if input_range is not None:
        if x.max() > 1 and input_range[1] == 1:
            x = x / 255.0
    if mean is not None:
        x = x - np.array(mean)
    if std is not None:
        x = x / np.array(std)
    return x


def get_preprocessing(mean, std, input_range=(0, 1)):
    """Tạo preprocessing transform (normalize + to_tensor)."""
    def _preprocess(image, **kwargs):
        return preprocess_input(image, mean=mean, std=std, input_range=input_range)

    return albu.Compose([
        albu.Lambda(image=_preprocess),
        albu.Lambda(image=to_tensor, mask=to_tensor),
    ])


# ===== Augmentation =====

def get_training_augmentation():
    return albu.Compose([
        albu.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.7),
        albu.GaussianBlur(blur_limit=(3, 7), p=0.3),
    ])


def get_validation_augmentation():
    return albu.Compose([
        albu.CenterCrop(height=256, width=256),
    ])


# ===== Tính Mean/Std từ tập train =====

def compute_mean_std(images_dir, is_rgb_only=IS_TRAINING_RGB_DATA):
    """Tính mean và std trên toàn bộ tập train."""
    image_files = [f for f in os.listdir(images_dir) if f.endswith(".tif")]
    means, stds = [], []
    for f in image_files:
        img = cv2.imread(os.path.join(images_dir, f), cv2.IMREAD_UNCHANGED)
        if is_rgb_only:
            img = img[:, :, :3]
        img = img / 255.0
        means.append(np.mean(img, axis=(0, 1)))
        stds.append(np.std(img, axis=(0, 1)))
    return np.mean(means, axis=0), np.mean(stds, axis=0)


# ===== Custom Dataset =====

class ForestSegmentationDataset(Dataset):
    """
    Dataset cho bài toán phân đoạn rừng.
    Hỗ trợ sliding window patches và augmentation.
    """
    def __init__(self, images_dir, masks_dir, file_list,
                 augmentation=None, preprocessing=None,
                 patch_size=None, stride=None):
        self.augmentation = augmentation
        self.preprocessing = preprocessing
        self.samples = []

        for image_name in sorted(file_list):
            if not image_name.endswith('.tif'):
                continue
            mask_name = image_name.replace("RGBNDVI", "MASK")
            image_path = os.path.join(images_dir, image_name)
            mask_path = os.path.join(masks_dir, mask_name)

            if not os.path.exists(mask_path):
                continue

            image, mask = load_image_and_mask(image_path, mask_path)
            mask = mask.astype(np.float32)

            if patch_size and stride:
                img_patches, mask_patches = sliding_window(image, mask, patch_size, stride)
                base, ext = os.path.splitext(image_name)
                for i, (img_p, msk_p) in enumerate(zip(img_patches, mask_patches)):
                    self.samples.append((img_p, msk_p, f"{base}_patch_{i+1}{ext}"))
            else:
                self.samples.append((image, mask, image_name))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image, mask, name = self.samples[idx]

        if self.augmentation:
            aug = self.augmentation(image=image, mask=mask)
            image, mask = aug['image'], aug['mask']

        if self.preprocessing:
            pre = self.preprocessing(image=image, mask=mask)
            image, mask = pre['image'], pre['mask']

        return torch.from_numpy(image), torch.from_numpy(mask), name


# ===== Factory: tạo DataLoader =====

def create_dataloaders(mean=None, std=None):
    """
    Tạo train_loader và valid_loader.
    Nếu mean/std chưa có, tự tính từ tập train.
    """
    if mean is None or std is None:
        print("Computing mean/std from training data...")
        mean, std = compute_mean_std(TRAIN_IMAGES_DIR)
        print(f"  MEAN: {mean}")
        print(f"  STD:  {std}")

    preprocessing = get_preprocessing(mean, std)

    train_dataset = ForestSegmentationDataset(
        images_dir=TRAIN_IMAGES_DIR,
        masks_dir=TRAIN_MASKS_DIR,
        file_list=os.listdir(TRAIN_IMAGES_DIR),
        augmentation=get_training_augmentation(),
        preprocessing=preprocessing,
        patch_size=PATCH_SIZE,
        stride=STRIDE,
    )

    valid_dataset = ForestSegmentationDataset(
        images_dir=VALID_IMAGES_DIR,
        masks_dir=VALID_MASKS_DIR,
        file_list=os.listdir(VALID_IMAGES_DIR),
        preprocessing=preprocessing,
        patch_size=PATCH_SIZE,
        stride=STRIDE,
    )

    g = torch.Generator()
    g.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, generator=g,
    )
    valid_loader = DataLoader(
        valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS,
    )

    print(f"Train samples: {len(train_dataset)}, Valid samples: {len(valid_dataset)}")
    return train_loader, valid_loader, mean, std


## 3. Loss Functions & Metrics

In [ ]:
"""
Loss functions & Metrics cho binary segmentation.
"""
import torch
import torch.nn as nn


# ===== Metrics =====

def dice_coefficient(pred, target, threshold=0.5, epsilon=1e-6):
    """Tính Dice coefficient (F1 cho segmentation)."""
    pred = (pred > threshold).float().view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return (2.0 * intersection + epsilon) / (pred.sum() + target.sum() + epsilon)


def compute_metrics(outputs, masks, threshold=0.5, epsilon=1e-6):
    """Tính IoU, Accuracy, Recall, Precision."""
    preds = (outputs > threshold).float().view(-1)
    masks = masks.view(-1)

    TP = (preds * masks).sum()
    TN = ((1 - preds) * (1 - masks)).sum()
    FP = (preds * (1 - masks)).sum()
    FN = ((1 - preds) * masks).sum()

    iou = (TP + epsilon) / (TP + FP + FN + epsilon)
    accuracy = (TP + TN + epsilon) / (TP + TN + FP + FN + epsilon)
    recall = (TP + epsilon) / (TP + FN + epsilon)
    precision = (TP + epsilon) / (TP + FP + epsilon)

    return iou.item(), accuracy.item(), recall.item(), precision.item()


# ===== Loss Functions =====

class DiceLoss(nn.Module):
    """Dice Loss = 1 - Dice coefficient."""
    def __init__(self, epsilon=1e-6):
        super().__init__()
        self.epsilon = epsilon

    def forward(self, pred, target):
        pred = pred.view(-1)
        target = target.view(-1)
        intersection = (pred * target).sum()
        dice = (2.0 * intersection + self.epsilon) / (pred.sum() + target.sum() + self.epsilon)
        return 1 - dice


class BCEDiceLoss(nn.Module):
    """Kết hợp BCE Loss + Dice Loss."""
    def __init__(self):
        super().__init__()
        self.bce = nn.BCELoss()
        self.dice = DiceLoss()

    def forward(self, pred, target):
        return self.bce(pred, target) + self.dice(pred, target)


## 4. Models

### U-Net

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def x2conv(in_channels, out_channels, inner_channels=None):
    """Khối tích chập kép (Conv-BN-ReLU) x2."""
    inner_channels = out_channels // 2 if inner_channels is None else inner_channels
    return nn.Sequential(
        nn.Conv2d(in_channels, inner_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(inner_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(inner_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

class Encoder(nn.Module):
    """Encoder block: x2conv + MaxPool."""
    def __init__(self, in_channels, out_channels):
        super(Encoder, self).__init__()
        self.down_conv = x2conv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, ceil_mode=True)

    def forward(self, x):
        x = self.down_conv(x)
        x = self.pool(x)
        return x

class Decoder(nn.Module):
    """Decoder block: Upsample + Concat + x2conv."""
    def __init__(self, in_channels, out_channels):
        super(Decoder, self).__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        # Khối tích chập xử lý kết quả sau khi nối với các đặc trưng từ encoder.
        self.up_conv = x2conv(in_channels, out_channels)

    def forward(self, skip, x, interpolate=True):
        x = self.up(x)
        if (x.size(2) != skip.size(2)) or (x.size(3) != skip.size(3)):
            if interpolate:
                x = F.interpolate(x, size=(skip.size(2), skip.size(3)), mode="bilinear", align_corners=True)
            else:
                diffY = skip.size()[2] - x.size()[2]
                diffX = skip.size()[3] - x.size()[3]
                x = F.pad(x, (diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2))
        
        x = torch.cat([skip, x], dim=1)
        x = self.up_conv(x)
        return x

class UNet(nn.Module):
    """
    Standard U-Net.
    Args:
        num_classes: số lớp đầu ra (1 cho binary segmentation).
        in_channels: số kênh đầu vào (3=RGB, 4=RGB+NDVI, 5=RGB+NDVI+Edge).
        freeze_bn: đóng băng BatchNorm hay không.
    """
    def __init__(self, num_classes=1, in_channels=4, freeze_bn=False):
        super(UNet, self).__init__()

        self.start_conv = x2conv(in_channels, 64)
        self.down1 = Encoder(64, 128)
        self.down2 = Encoder(128, 256)
        self.down3 = Encoder(256, 512)
        self.down4 = Encoder(512, 1024)

        self.middle_conv = x2conv(1024, 1024)

        self.up1 = Decoder(1024, 512)
        self.up2 = Decoder(512, 256)
        self.up3 = Decoder(256, 128)
        self.up4 = Decoder(128, 64)

        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)

        self._initialize_weights()
        if freeze_bn:
            self.freeze_bn()

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(module.weight)
                if module.bias is not None:
                    module.bias.data.zero_()
            elif isinstance(module, nn.BatchNorm2d):
                module.weight.data.fill_(1)
                module.bias.data.zero_()

    def forward(self, x):
        x1 = self.start_conv(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x_mid = self.middle_conv(x5)

        x = self.up1(x4, x_mid)
        x = self.up2(x3, x)
        x = self.up3(x2, x)
        x = self.up4(x1, x)

        x = self.final_conv(x)
        return torch.sigmoid(x)

    def freeze_bn(self):
        for module in self.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()


### Attention U-Net

In [ ]:
"""
Attention U-Net: U-Net với Attention Gates ở skip connections.
Reference: Oktay et al., "Attention U-Net: Learning Where to Look for the Pancreas", 2018.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def x2conv(in_channels, out_channels, inner_channels=None):
    """Khối tích chập đôi (Conv-BN-ReLU) x2."""
    inner_channels = out_channels // 2 if inner_channels is None else inner_channels
    return nn.Sequential(
        nn.Conv2d(in_channels, inner_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(inner_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(inner_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


class AttentionGate(nn.Module):
    """
    Attention Gate: lọc thông tin skip connection bằng tín hiệu gating từ decoder.
    - x: skip feature từ encoder (batch, C_x, H, W)
    - g: gating feature từ decoder (batch, C_g, h, w)
    """
    def __init__(self, in_channels, gating_channels, inter_channels=None):
        super().__init__()
        if inter_channels is None:
            inter_channels = in_channels // 2

        self.W_x = nn.Conv2d(in_channels, inter_channels, kernel_size=1, bias=True)
        self.W_g = nn.Conv2d(gating_channels, inter_channels, kernel_size=1, bias=True)
        self.psi = nn.Sequential(
            nn.Conv2d(inter_channels, 1, kernel_size=1, bias=True),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, g):
        x1 = self.W_x(x)
        g1 = self.W_g(g)

        if x1.shape[2:] != g1.shape[2:]:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode='bilinear', align_corners=True)

        f = self.relu(x1 + g1)
        psi = self.psi(f)
        return x * psi


class Encoder(nn.Module):
    """Encoder block: x2conv + MaxPool."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.down_conv = x2conv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, ceil_mode=True)

    def forward(self, x):
        x = self.down_conv(x)
        x_pooled = self.pool(x)
        return x, x_pooled  # skip feature, pooled output


class Decoder(nn.Module):
    """Decoder block: Upsample + Attention Gate + Concat + x2conv."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.att_gate = AttentionGate(
            in_channels=in_channels,
            gating_channels=in_channels // 2,
        )
        self.up_conv = x2conv(in_channels + in_channels // 2, out_channels)

    def forward(self, skip_feat, x):
        x = self.up(x)
        if x.shape[2:] != skip_feat.shape[2:]:
            x = F.interpolate(x, size=skip_feat.shape[2:], mode='bilinear', align_corners=True)

        skip_att = self.att_gate(skip_feat, x)
        x = torch.cat([skip_att, x], dim=1)
        x = self.up_conv(x)
        return x


class AttentionUNet(nn.Module):
    """
    Attention U-Net.
    Args:
        num_classes: số lớp đầu ra (1 cho binary segmentation).
        in_channels: số kênh đầu vào (3=RGB, 4=RGB+NDVI).
        freeze_bn: đóng băng BatchNorm hay không.
    """
    def __init__(self, num_classes=1, in_channels=4, freeze_bn=False):
        super().__init__()

        # Encoder
        self.start_conv = x2conv(in_channels, 64)
        self.enc1 = Encoder(64, 128)
        self.enc2 = Encoder(128, 256)
        self.enc3 = Encoder(256, 512)
        self.enc4 = Encoder(512, 1024)

        # Bottleneck
        self.bottleneck = x2conv(1024, 1024)

        # Decoder
        self.dec1 = Decoder(1024, 512)
        self.dec2 = Decoder(512, 256)
        self.dec3 = Decoder(256, 128)
        self.dec4 = Decoder(128, 64)

        # Final
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)

        self._initialize_weights()
        if freeze_bn:
            self.freeze_bn()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def forward(self, x):
        x1 = self.start_conv(x)
        skip1, x2p = self.enc1(x1)
        skip2, x3p = self.enc2(x2p)
        skip3, x4p = self.enc3(x3p)
        skip4, x5p = self.enc4(x4p)

        x_b = self.bottleneck(x5p)

        x = self.dec1(skip4, x_b)
        x = self.dec2(skip3, x)
        x = self.dec3(skip2, x)
        x = self.dec4(skip1, x)

        return torch.sigmoid(self.final_conv(x))

    def freeze_bn(self):
        for m in self.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()


### CBAM U-Net

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()
        avg_pool = F.adaptive_avg_pool2d(x, 1).view(b, c)
        max_pool = F.adaptive_max_pool2d(x, 1).view(b, c)
        attn = self.sigmoid(self.mlp(avg_pool) + self.mlp(max_pool))
        return x * attn.view(b, c, 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size,
                              padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_pool = torch.mean(x, dim=1, keepdim=True)
        max_pool, _ = torch.max(x, dim=1, keepdim=True)
        cat = torch.cat([avg_pool, max_pool], dim=1)
        attn = self.sigmoid(self.conv(cat))
        return x * attn

class CBAM(nn.Module):
    """Convolutional Block Attention Module"""
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attn = ChannelAttention(in_channels, reduction)
        self.spatial_attn = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_attn(x)
        x = self.spatial_attn(x)
        return x

class ChannelAttentionPriority(nn.Module):
    """Channel Attention with Priority for specific channels (e.g., NDVI)"""
    def __init__(self, in_channels, reduction=16, ndvi_idx=3, ndvi_factor=2.0):
        super().__init__()
        self.ndvi_idx = ndvi_idx         # index của kênh NDVI, 0-based
        self.ndvi_factor = ndvi_factor   # hệ số ưu tiên
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()
        # 1) Global Pooling Avg & Max
        avg_pool = F.adaptive_avg_pool2d(x, 1).view(b, c)
        max_pool = F.adaptive_max_pool2d(x, 1).view(b, c)

        # 2) Ưu tiên NDVI
        avg_pool_clone = avg_pool.clone()
        max_pool_clone = max_pool.clone()
        avg_pool_clone[:, self.ndvi_idx] = avg_pool_clone[:, self.ndvi_idx] * self.ndvi_factor
        max_pool_clone[:, self.ndvi_idx] = max_pool_clone[:, self.ndvi_idx] * self.ndvi_factor

        # 3) Qua MLP & Sigmoid
        attn = self.sigmoid(self.mlp(avg_pool_clone) + self.mlp(max_pool_clone))
        return x * attn.view(b, c, 1, 1)

class CBAMPriority(nn.Module):
    """CBAM with Channel Attention Priority"""
    def __init__(self, in_channels, reduction=16, kernel_size=7, ndvi_idx=3, ndvi_factor=2.0):
        super().__init__()
        self.channel_attn = ChannelAttentionPriority(in_channels, reduction, ndvi_idx, ndvi_factor)
        self.spatial_attn = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_attn(x)
        x = self.spatial_attn(x)
        return x

def x2conv(in_channels, out_channels, inner_channels=None, use_cbam=False, priority=False):
    inner_channels = out_channels // 2 if inner_channels is None else inner_channels
    layers = [
        nn.Conv2d(in_channels, inner_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(inner_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(inner_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    ]
    if use_cbam:
        if priority:
            layers.append(CBAMPriority(out_channels))
        else:
            layers.append(CBAM(out_channels))
    return nn.Sequential(*layers)

class Encoder(nn.Module):
    def __init__(self, in_channels, out_channels, use_cbam=False, priority=False):
        super().__init__()
        self.conv = x2conv(in_channels, out_channels, use_cbam=use_cbam, priority=priority)
        self.pool = nn.MaxPool2d(2, ceil_mode=True)

    def forward(self, x):
        skip = self.conv(x)
        x_pooled = self.pool(skip)
        return skip, x_pooled

class Decoder(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, use_cbam=False, priority=False):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        total_channels = (in_channels // 2) + skip_channels
        self.up_conv = x2conv(total_channels, out_channels, use_cbam=use_cbam, priority=priority)

    def forward(self, skip, x):
        x = self.up(x)
        if x.size()[-2:] != skip.size()[-2:]:
            x = F.interpolate(x, size=skip.size()[-2:], mode='bilinear', align_corners=True)
        x = torch.cat([skip, x], dim=1)
        x = self.up_conv(x)
        return x

class UNetCBAM(nn.Module):
    """
    U-Net with Convolutional Block Attention Module (CBAM)
    """
    def __init__(self, num_classes=1, in_channels=4, freeze_bn=False, use_cbam=True):
        super().__init__()
        self.start_conv = x2conv(in_channels, 64, use_cbam=use_cbam)
        self.enc1 = Encoder(64, 128, use_cbam=use_cbam)
        self.enc2 = Encoder(128, 256, use_cbam=use_cbam)
        self.enc3 = Encoder(256, 512, use_cbam=use_cbam)
        self.enc4 = Encoder(512, 1024, use_cbam=use_cbam)

        self.bottleneck = x2conv(1024, 1024, use_cbam=use_cbam)

        self.dec1 = Decoder(1024, skip_channels=1024, out_channels=512, use_cbam=use_cbam)
        self.dec2 = Decoder(512, skip_channels=512, out_channels=256, use_cbam=use_cbam)
        self.dec3 = Decoder(256, skip_channels=256, out_channels=128, use_cbam=use_cbam)
        self.dec4 = Decoder(128, skip_channels=128, out_channels=64, use_cbam=use_cbam)

        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)
        self._initialize_weights()
        if freeze_bn:
            self.freeze_bn()

    def forward(self, x):
        x0 = self.start_conv(x)
        s1, x1 = self.enc1(x0)
        s2, x2 = self.enc2(x1)
        s3, x3 = self.enc3(x2)
        s4, x4 = self.enc4(x3)

        x_mid = self.bottleneck(x4)

        x = self.dec1(s4, x_mid)
        x = self.dec2(s3, x)
        x = self.dec3(s2, x)
        x = self.dec4(s1, x)

        return torch.sigmoid(self.final_conv(x))

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def freeze_bn(self):
        for m in self.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()

class UNetCBAMPriority(UNetCBAM):
    """
    U-Net with CBAM and Priority on specific channel (e.g. NDVI).
    """
    def __init__(self, num_classes=1, in_channels=4, freeze_bn=False, use_cbam=True):
        nn.Module.__init__(self)
        self.start_conv = x2conv(in_channels, 64, use_cbam=use_cbam, priority=True)
        self.enc1 = Encoder(64, 128, use_cbam=use_cbam, priority=True)
        self.enc2 = Encoder(128, 256, use_cbam=use_cbam, priority=True)
        self.enc3 = Encoder(256, 512, use_cbam=use_cbam, priority=True)
        self.enc4 = Encoder(512, 1024, use_cbam=use_cbam, priority=True)

        self.bottleneck = x2conv(1024, 1024, use_cbam=use_cbam, priority=True)

        self.dec1 = Decoder(1024, skip_channels=1024, out_channels=512, use_cbam=use_cbam, priority=True)
        self.dec2 = Decoder(512, skip_channels=512, out_channels=256, use_cbam=use_cbam, priority=True)
        self.dec3 = Decoder(256, skip_channels=256, out_channels=128, use_cbam=use_cbam, priority=True)
        self.dec4 = Decoder(128, skip_channels=128, out_channels=64, use_cbam=use_cbam, priority=True)

        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)
        self._initialize_weights()
        if freeze_bn:
            self.freeze_bn()


## 5. Training Loop

In [ ]:
import os
import time
import csv
import torch
import torch.optim as optim

# from config import (
    NUM_EPOCHS, LEARNING_RATE, PATIENCE, SEED, CHECKPOINT_DIR,
    IS_TRAINING_RGB_DATA, NUM_IN_CHANNELS, NUM_CLASSES
)
# from dataset import create_dataloaders
# from losses import BCEDiceLoss, dice_coefficient, compute_metrics

# Models
# from models.unet import UNet
# from models.attention_unet import AttentionUNet
# from models.cbam_unet import UNetCBAM, UNetCBAMPriority

def train_model(model, num_epochs, train_loader, valid_loader, device, model_name, patience=5):
    type_data = 'RGB' if IS_TRAINING_RGB_DATA else 'RGB_NDVI'
    print(f'Training model {model_name} ({type_data}) with {device}...')

    criterion = BCEDiceLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    model.to(device)

    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    best_val_dice = 0.0
    counter = 0

    log_path = os.path.join(CHECKPOINT_DIR, f"training_log_{model_name}_{time.strftime('%Y%m%d_%H%M%S')}.csv")
    
    with open(log_path, "w", newline="") as csvfile:
        fieldnames = [
            "epoch",
            "train_loss", "train_dice", "train_iou", "train_accuracy", "train_recall", "train_precision",
            "val_loss", "val_dice", "val_iou", "val_accuracy", "val_recall", "val_precision"
        ]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for epoch in range(1, num_epochs + 1):
            model.train()
            train_stats = {k: 0.0 for k in fieldnames[1:7]}
            for images, masks, _ in train_loader:
                images, masks = images.to(device), masks.to(device)
                if masks.dim() == 3:
                    masks = masks.unsqueeze(1)

                outputs = model(images)
                loss = criterion(outputs, masks)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                d = dice_coefficient(outputs, masks)
                iou, acc, rec, prec = compute_metrics(outputs, masks)
                
                train_stats["train_loss"] += loss.item()
                train_stats["train_dice"] += d.item() if isinstance(d, torch.Tensor) else d
                train_stats["train_iou"] += iou
                train_stats["train_accuracy"] += acc
                train_stats["train_recall"] += rec
                train_stats["train_precision"] += prec

            batches = len(train_loader)
            for k in train_stats:
                train_stats[k] /= batches

            model.eval()
            val_stats = {k: 0.0 for k in fieldnames[7:]}
            with torch.no_grad():
                for images, masks, _ in valid_loader:
                    images, masks = images.to(device), masks.to(device)
                    if masks.dim() == 3:
                        masks = masks.unsqueeze(1)

                    outputs = model(images)
                    loss = criterion(outputs, masks)

                    d = dice_coefficient(outputs, masks)
                    iou, acc, rec, prec = compute_metrics(outputs, masks)
                    
                    val_stats["val_loss"] += loss.item()
                    val_stats["val_dice"] += d.item() if isinstance(d, torch.Tensor) else d
                    val_stats["val_iou"] += iou
                    val_stats["val_accuracy"] += acc
                    val_stats["val_recall"] += rec
                    val_stats["val_precision"] += prec

            vbatches = len(valid_loader)
            for k in val_stats:
                val_stats[k] /= vbatches

            row = {"epoch": epoch}
            row.update(train_stats)
            row.update(val_stats)
            writer.writerow(row)
            csvfile.flush()

            print(f"Epoch {epoch}/{num_epochs} - Train Dice: {train_stats['train_dice']:.4f} | Val Dice: {val_stats['val_dice']:.4f}")

            if val_stats["val_dice"] > best_val_dice:
                best_val_dice = val_stats["val_dice"]
                counter = 0
                ckpt_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_best.pth")
                torch.save(model.state_dict(), ckpt_path)
                print(f"  -> New best model saved at epoch {epoch}")
            else:
                counter += 1
                if counter >= patience:
                    print(f"Early stopping: khong cai thien sau {patience} epochs.")
                    break

    print("Training ket thuc. Best Val Dice:", best_val_dice)
    return ckpt_path

def main():
    torch.manual_seed(SEED)
    
    # Giới hạn số luồng CPU mà PyTorch sử dụng để tránh full 100% CPU
    # torch.set_num_threads(2)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load dataloaders
    train_loader, valid_loader, mean, std = create_dataloaders()
    
    print("\n--- Training Attention U-Net ---")
    model_attention = AttentionUNet(num_classes=NUM_CLASSES, in_channels=NUM_IN_CHANNELS)
    train_model(model_attention, NUM_EPOCHS, train_loader, valid_loader, device, "AttentionUnet", PATIENCE)

    # Nếu muốn train các mô hình khác, bạn có thể bỏ comment dưới đây:
    # print("\n--- Training U-Net ---")
    # model_unet = UNet(num_classes=NUM_CLASSES, in_channels=NUM_IN_CHANNELS)
    # train_model(model_unet, NUM_EPOCHS, train_loader, valid_loader, device, "UNet", PATIENCE)

    # print("\n--- Training UNet-CBAM ---")
    # model_cbam = UNetCBAM(num_classes=NUM_CLASSES, in_channels=NUM_IN_CHANNELS)
    # train_model(model_cbam, NUM_EPOCHS, train_loader, valid_loader, device, "UNetCBAM", PATIENCE)

if __name__ == "__main__":
    main()
